In [0]:
dbutils.widgets.text("dataset_name", "", "Gold Dataset");

dataset_name = dbutils.widgets.get("dataset_name").strip();

In [0]:
# ============================================================
# 03_Gold_Transformation
# Parameterized Gold Transformation
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

if not dataset_name:
    raise ValueError(
        "dataset_name parameter is required."
    )

print("=" * 80)
print(f"Starting Gold Transformation: {dataset_name}")
print("=" * 80)


# ============================================================
# 2. CATALOG / SCHEMA
# ============================================================

CATALOG = "f1_catalog"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"


# ============================================================
# 3. SOURCE TABLES
# ============================================================

drivers_table = f"{CATALOG}.{SILVER_SCHEMA}.drivers"

constructors_table = (
    f"{CATALOG}.{SILVER_SCHEMA}.constructors"
)

races_table = f"{CATALOG}.{SILVER_SCHEMA}.races"

results_table = f"{CATALOG}.{SILVER_SCHEMA}.results"

driver_standings_table = (
    f"{CATALOG}.{SILVER_SCHEMA}.driver_standings"
)

constructor_standings_table = (
    f"{CATALOG}.{SILVER_SCHEMA}.constructor_standings"
)

circuits_table = (
    f"{CATALOG}.{SILVER_SCHEMA}.circuits"
)


# ============================================================
# 4. DRIVER PERFORMANCE
# ============================================================

if dataset_name == "driver_performance":

    print("Creating Driver Performance...")


    results = spark.table(results_table)

    drivers = spark.table(drivers_table)

    constructors = spark.table(
        constructors_table
    )

    races = spark.table(races_table)


    # --------------------------------------------------------
    # Join Results + Drivers
    # --------------------------------------------------------

    df = (
        results.alias("r")
        .join(
            drivers.alias("d"),
            F.col("r.driver_id")
            == F.col("d.driver_id"),
            "left"
        )
    )


    # --------------------------------------------------------
    # Join Constructors
    # --------------------------------------------------------

    df = (
        df.join(
            constructors.alias("c"),
            F.col("r.constructor_id")
            == F.col("c.constructor_id"),
            "left"
        )
    )


    # --------------------------------------------------------
    # Join Races
    # --------------------------------------------------------

    df = (
        df.join(
            races.alias("ra"),
            F.col("r.race_id")
            == F.col("ra.race_id"),
            "left"
        )
    )


    # --------------------------------------------------------
    # Aggregate Driver Performance
    # --------------------------------------------------------

    df = (
        df.groupBy(
            F.col("r.driver_id"),
            F.col("d.first_name"),
            F.col("d.last_name")
        )
        .agg(

            F.countDistinct(
                "r.race_id"
            ).alias("races"),

            F.sum(
                "r.points"
            ).alias("total_points"),

            F.sum(
                F.when(
                    F.col("r.finish_position") == 1,
                    1
                ).otherwise(0)
            ).alias("race_wins"),

            F.sum(
                F.when(
                    F.col("r.finish_position") <= 3,
                    1
                ).otherwise(0)
            ).alias("podiums"),

            F.avg(
                "r.finish_position"
            ).alias("avg_finish_position"),

            F.avg(
                "r.grid"
            ).alias("avg_grid_position")
        )
    )


    # --------------------------------------------------------
    # Driver Ranking
    # --------------------------------------------------------

    ranking_window = Window.orderBy(
        F.desc("total_points")
    )


    df = df.withColumn(
        "driver_rank",
        F.dense_rank().over(
            ranking_window
        )
    )


    # --------------------------------------------------------
    # Driver Name
    # --------------------------------------------------------

    df = df.withColumn(
        "driver_name",
        F.concat_ws(
            " ",
            "first_name",
            "last_name"
        )
    )


    # --------------------------------------------------------
    # Gold Metadata
    # --------------------------------------------------------

    df = (
        df
        .withColumn(
            "_gold_processed_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_gold_dataset",
            F.lit("driver_performance")
        )
    )


    target_table = (
        f"{CATALOG}.{GOLD_SCHEMA}."
        f"driver_performance"
    )


# ============================================================
# 5. CONSTRUCTOR PERFORMANCE
# ============================================================

elif dataset_name == "constructor_performance":

    print("Creating Constructor Performance...")


    results = spark.table(
        results_table
    )

    constructors = spark.table(
        constructors_table
    )


    # --------------------------------------------------------
    # Join Results + Constructors
    # --------------------------------------------------------

    df = (
        results.alias("r")
        .join(
            constructors.alias("c"),
            F.col("r.constructor_id")
            == F.col("c.constructor_id"),
            "left"
        )
    )


    # --------------------------------------------------------
    # Aggregate
    # --------------------------------------------------------

    df = (
        df.groupBy(
            F.col("r.constructor_id"),
            F.col("c.constructor_name")
        )
        .agg(

            F.countDistinct(
                "r.race_id"
            ).alias("races"),

            F.sum(
                "r.points"
            ).alias("total_points"),

            F.sum(
                F.when(
                    F.col("r.finish_position") == 1,
                    1
                ).otherwise(0)
            ).alias("race_wins"),

            F.sum(
                F.when(
                    F.col("r.finish_position") <= 3,
                    1
                ).otherwise(0)
            ).alias("podiums"),

            F.avg(
                "r.finish_position"
            ).alias("avg_finish_position")
        )
    )


    # --------------------------------------------------------
    # Ranking
    # --------------------------------------------------------

    ranking_window = Window.orderBy(
        F.desc("total_points")
    )


    df = df.withColumn(
        "constructor_rank",
        F.dense_rank().over(
            ranking_window
        )
    )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    df = (
        df
        .withColumn(
            "_gold_processed_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_gold_dataset",
            F.lit("constructor_performance")
        )
    )


    target_table = (
        f"{CATALOG}.{GOLD_SCHEMA}."
        f"constructor_performance"
    )


# ============================================================
# 6. SEASON SUMMARY
# ============================================================

elif dataset_name == "season_summary":

    print("Creating Season Summary...")


    results = spark.table(
        results_table
    )

    races = spark.table(
        races_table
    )


    # --------------------------------------------------------
    # Join Results + Races
    # --------------------------------------------------------

    df = (
        results.alias("r")
        .join(
            races.alias("ra"),
            F.col("r.race_id")
            == F.col("ra.race_id"),
            "left"
        )
    )


    # --------------------------------------------------------
    # Season Aggregation
    # --------------------------------------------------------

    df = (
        df.groupBy(
            F.col("ra.season")
        )
        .agg(

            F.countDistinct(
                "r.race_id"
            ).alias("total_races"),

            F.countDistinct(
                "r.driver_id"
            ).alias("total_drivers"),

            F.countDistinct(
                "r.constructor_id"
            ).alias("total_constructors"),

            F.sum(
                "r.points"
            ).alias("total_points"),

            F.sum(
                F.when(
                    F.col("r.finish_position") == 1,
                    1
                ).otherwise(0)
            ).alias("total_wins")
        )
    )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    df = (
        df
        .withColumn(
            "_gold_processed_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_gold_dataset",
            F.lit("season_summary")
        )
    )


    target_table = (
        f"{CATALOG}.{GOLD_SCHEMA}."
        f"season_summary"
    )


# ============================================================
# 7. CIRCUIT PERFORMANCE
# ============================================================

elif dataset_name == "circuit_performance":

    print("Creating Circuit Performance...")


    results = spark.table(
        results_table
    )

    races = spark.table(
        races_table
    )

    circuits = spark.table(
        circuits_table
    )


    # --------------------------------------------------------
    # Results + Races
    # --------------------------------------------------------

    df = (
        results.alias("r")
        .join(
            races.alias("ra"),
            F.col("r.race_id")
            == F.col("ra.race_id"),
            "left"
        )
    )


    # --------------------------------------------------------
    # + Circuits
    # --------------------------------------------------------

    df = (
        df.join(
            circuits.alias("c"),
            F.col("ra.circuit_id")
            == F.col("c.circuit_id"),
            "left"
        )
    )


    # --------------------------------------------------------
    # Aggregate Circuit Performance
    # --------------------------------------------------------

    df = (
        df.groupBy(
            F.col("c.circuit_id"),
            F.col("c.circuit_name")
        )
        .agg(

            F.countDistinct(
                "r.race_id"
            ).alias("total_races"),

            F.avg(
                "r.finish_position"
            ).alias("avg_finish_position"),

            F.sum(
                "r.points"
            ).alias("total_points"),

            F.sum(
                F.when(
                    F.col("r.finish_position") == 1,
                    1
                ).otherwise(0)
            ).alias("total_wins")
        )
    )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    df = (
        df
        .withColumn(
            "_gold_processed_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_gold_dataset",
            F.lit("circuit_performance")
        )
    )


    target_table = (
        f"{CATALOG}.{GOLD_SCHEMA}."
        f"circuit_performance"
    )


# ============================================================
# 8. INVALID DATASET
# ============================================================

else:

    raise ValueError(
        f"Unsupported Gold dataset: "
        f"{dataset_name}"
    )


# ============================================================
# 9. DISPLAY RESULT
# ============================================================

print("\nGold Schema:")

df.printSchema()

display(
    df.limit(20)
)


# ============================================================
# 10. WRITE GOLD DELTA TABLE
# ============================================================

print(
    f"\nWriting to: {target_table}"
)


(
    df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        target_table
    )
)


# ============================================================
# 11. VALIDATE
# ============================================================

gold_count = spark.table(
    target_table
).count()


print("\n" + "=" * 80)

print("GOLD TRANSFORMATION SUCCESSFUL")

print("=" * 80)

print(
    f"Dataset      : {dataset_name}"
)

print(
    f"Target table : {target_table}"
)

print(
    f"Gold records : {gold_count}"
)

print("=" * 80)


# ============================================================
# 12. RETURN RESULT TO PARENT
# ============================================================

dbutils.notebook.exit(
    f"SUCCESS | {dataset_name} | "
    f"{gold_count} records"
)